# Contextual Chunk Headers（CCH）：给 chunk 加“上位上下文”再去 embedding

本节目标：实现 CCH 的最小版本，并理解它为什么能显著提升检索质量。

## 核心直觉

很多 chunk 单独拿出来时缺少“指代对象/所属文档/章节主题”等上位信息：

- chunk 里可能只写“we / the company / this report / it”，没有明确实体
- chunk 可能需要“文档标题/章节标题/小节标题”才能被正确理解

CCH 的做法是：

1. 构造一个 chunk header（例如：`Document Title: ...`，可选加摘要/章节层级）
2. 在**embedding 前**把 header 拼到 chunk 文本前面
3. 在**给 LLM 展示检索结果**时，也把 header 一起展示

这样 embeddings/LLM 都能拿到更完整的语义线索。

## 0) 导入依赖并加载环境变量

延续前面 notebook 的约定：从 `../.env` 读取 `OPENAI_API_KEY`。

In [1]:
from __future__ import annotations

import sys
from pathlib import Path
from typing import List

import numpy as np
import requests
from dotenv import load_dotenv

load_dotenv("../.env")

# 让 `all_rag_techniques/` 子目录里的 notebook
# 也能导入上一级目录的 `helper_functions.py` / `evaluation/`
sys.path.insert(0, str(Path("..").resolve()))

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS

from helper_functions import show_context

/tmp/ipykernel_3260379/2334690972.py:21: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


## 1) 准备示例文档（Nike 10-K 文本）并切 chunks

原仓库用 `nike_2023_annual_report.txt` 来做演示：它非常适合展示“chunk 没写 Nike，但其实属于 Nike 报告”的典型问题。

如果你本地还没有该文件，这里会直接下载到 `../data/`（失败会直接抛错，便于你排查网络/权限问题）。

In [2]:
import os
os.environ["HTTP_PROXY"] = "http://127.0.0.1:7890"
os.environ["HTTPS_PROXY"] = "http://127.0.0.1:7890"
os.environ["ALL_PROXY"] = "http://127.0.0.1:7890"

In [3]:
DATA_DIR = Path("../data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

NIKE_URL = "https://raw.githubusercontent.com/NirDiamant/RAG_Techniques/main/data/nike_2023_annual_report.txt"
NIKE_PATH = DATA_DIR / "nike_2023_annual_report.txt"

if not NIKE_PATH.exists():
    resp = requests.get(NIKE_URL, timeout=60)
    resp.raise_for_status()
    NIKE_PATH.write_bytes(resp.content)

NIKE_PATH

PosixPath('../data/nike_2023_annual_report.txt')

In [4]:
document_text = NIKE_PATH.read_text(encoding="utf-8", errors="ignore")

splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=0, length_function=len)
docs = splitter.create_documents([document_text])
chunks = [d.page_content for d in docs]

len(chunks), chunks[0][:200]

(500,
 'FORM 10-K FORM 10-KUNITED STATES\nSECURITIES AND EXCHANGE COMMISSION\nWashington, D.C.\nWashington, D.C. 20549\nFORM 10-K \n(Mark One)\n☑ ANNUAL REPORT PURSUANT TO SECTION 13 OR 15(D) OF THE SECURITIES EXCH')

## 2) 生成文档标题（用作 chunk header 的最小上位上下文）

原仓库的经验结论是：

- **文档标题**往往是最便宜、最有效的一类“上位上下文”
- 如果你的数据源本身已经有可靠标题（例如数据库字段、文件名、网页标题），可以直接用，不一定要 LLM 生成

这里我们用 LLM 从文档开头截取一段来生成标题。

In [5]:
llm_title = ChatOpenAI(model="gpt-4o-mini", temperature=0.2)

prompt_title = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "你是一个文档标题提取器。你的输出必须只有标题本身，不要输出任何额外内容。",
        ),
        (
            "user",
            "请为下面这份文档生成一个简洁、描述性强的标题。\n\n"
            "注意：我提供的只是文档开头的一部分，但你的标题应当概括整份文档。\n\n"
            "文档内容（截断）：\n{snippet}",
        ),
    ]
)

snippet = document_text[:12000]
document_title = (prompt_title | llm_title).invoke({"snippet": snippet}).content.strip()

print(document_title)

NIKE, Inc. Annual Report on Form 10-K for Fiscal Year Ended May 31, 2023


## 3) 只加一个 header，就能让“问法”和“chunk语义”更对齐（最小示例）

我们复现原仓库的核心现象：

- chunk 本身在讲“某公司受到气候变化影响”，但不一定写出公司名
- 当 query 显式包含公司名（例如 `Nike climate change impact`）时
  - **不加 header**：chunk 与 query 的相似度可能很低
  - **加上 `Document Title: ...`**：相似度会显著提高

原仓库用 Cohere reranker 做打分；为了在本地环境更容易跑通，这里用 **OpenAI embeddings 的 cosine similarity** 来演示同一个现象。

In [6]:
def cosine_similarity(vec1, vec2) -> float:
    v1 = np.asarray(vec1, dtype=np.float32)
    v2 = np.asarray(vec2, dtype=np.float32)
    return float(np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2)))


emb = OpenAIEmbeddings(model="text-embedding-3-small")

CHUNK_INDEX_TO_INSPECT = 86  # 与原仓库一致的示例索引（如需可自行调整）
query = "Nike climate change impact"

chunk_text = chunks[CHUNK_INDEX_TO_INSPECT]
chunk_wo_header = chunk_text
chunk_w_header = f"Document Title: {document_title}\n\n{chunk_text}"

q_vec = emb.embed_query(query)
wo_vec = emb.embed_documents([chunk_wo_header])[0]
w_vec = emb.embed_documents([chunk_w_header])[0]

print("Chunk header:")
print(f"Document Title: {document_title}")
print("\nChunk text (first 400 chars):")
print(chunk_text[:400])
print("\nQuery:", query)
print("\nCosine similarity w/o header:", cosine_similarity(q_vec, wo_vec))
print("Cosine similarity with header:", cosine_similarity(q_vec, w_vec))

Chunk header:
Document Title: NIKE, Inc. Annual Report on Form 10-K for Fiscal Year Ended May 31, 2023

Chunk text (first 400 chars):
Given the broad and global scope of our operations, we are particularly vulnerable to the physical risks of climate change, such 
as shifts in weather patterns. Extreme weather conditions in the areas in which our retail stores, suppliers, manufacturers, 
customers, distribution centers, offices, headquarters and vendors are located could adversely affect our operating results and 
financial condi

Query: Nike climate change impact

Cosine similarity w/o header: 0.36846816539764404
Cosine similarity with header: 0.49665090441703796


## 4) 真正用于检索：把 header 拼到每个 chunk 上再建库

现在我们把 CCH 用到整个语料：

- 每个 chunk 的 embedding 输入变成：`header + "\n\n" + chunk`
- 检索返回的 `page_content` 也带 header（对下游 LLM 更友好）

In [7]:
header = f"Document Title: {document_title}" 

cch_docs: List[Document] = []
for i, c in enumerate(chunks):
    content = f"{header}\n\n{c}"
    cch_docs.append(
        Document(
            page_content=content,
            metadata={"doc_title": document_title, "chunk_id": i},
        )
    )

vectorstore = FAISS.from_documents(cch_docs, OpenAIEmbeddings(model="text-embedding-3-small"))
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

retriever

VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x7dec89891b20>, search_kwargs={'k': 3})

## 5) 试一下检索结果

观察检索出来的 chunk 是否因为 header 而“更容易被命中/更容易被理解”。

In [8]:
query = "Nike climate change impact"
docs = retriever.invoke(query)

show_context([d.page_content for d in docs])

Context 1:
Document Title: NIKE, Inc. Annual Report on Form 10-K for Fiscal Year Ended May 31, 2023

increasingly frequent and/or prolonged extreme weather and climate events. Climate change may also exacerbate challenges 
relating to the availability and quality of water and raw materials, including those used in the production of our products, and may 
result in changes in regulations or consumer preferences, which could in turn af fect our business, operating results and financial 
condition. For example, there has been increased focus by governmental and non-governmental organizations, consumers, 
customers, employees and other stakeholders on products that are sustainably made and other sustainability matters, including 
responsible sourcing and deforestation, the use of plastic, energy and water , the recyclability or recoverability of packaging and

Context 2:
Document Title: NIKE, Inc. Annual Report on Form 10-K for Fiscal Year Ended May 31, 2023

could have an adverse impact o

## 6) 小结：CCH 适合什么时候用？

- chunk 里经常出现**代词/隐含指代**（we/it/this report/本公司/该产品）
- chunk 单独看会“失去主体”，导致检索和阅读都困难
- 你想用最小成本提升检索（比起重写 chunk、重做复杂切分，header 往往很便宜）

原仓库还给出了更大规模评测（KITE、FinanceBench）显示总体增益。你可以把它理解为一种非常实用的“上下文补全”工程技巧：

- embedding 输入变得更完整
- 下游 LLM 的引用与理解也更稳